# AgentComet — Complete Feature Guide

This notebook demonstrates **all** features of the AgentComet SDK:

1. **Tools** — `@tool` decorator + builtin tools
2. **Agent Class** — Custom agents with `setup()`
3. **Memory & Conversations** — Auto-saved chat history + key-value store
4. **State Persistence** — Named checkpoints with rollback
5. **UAF Export & Load** — Portable agents that remember everything
6. **Local Server** — Seamlessly push/pull agents \n
7. **LLM Providers** — Ollama, OpenAI, Gemini, Anthropic, etc.

**Prerequisites:**
```bash
pip install uaf pyyaml requests
pip install -e .  # Install agentcomet
```

---

## Setup

In [2]:
import os
import shutil

# Load .env file manually to support zero-dependency environment loading
if os.path.exists(".env"):
    with open(".env", "r") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, _, v = line.partition("=")
                os.environ[k.strip()] = v.strip()

from agentcomet import Agent, create_agent, load_agent
from agentcomet.models import Ollama
from agentcomet.tools import tool

llm = Ollama(model="gemma3:4b")
print("LLM ready:", llm.model)

LLM ready: gemma3:4b


---

## 1. Tools

The `@tool` decorator converts any function into a `ToolSpec` with auto-extracted name, description, and JSON schema.

In [3]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two numbers together."""
    return a * b

@tool
def calculator(expression: str) -> str:
    """Evaluates a mathematical expression."""
    return str(eval(expression))

print("Name:", multiply.name)
print("Description:", multiply.description)
print("Schema:", multiply.schema)

Name: multiply
Description: Multiplies two numbers together.
Schema: {'type': 'object', 'properties': {'a': {'type': 'integer', 'description': 'Parameter a'}, 'b': {'type': 'integer', 'description': 'Parameter b'}}, 'required': ['a', 'b']}


In [4]:
# Builtin tools
from agentcomet.tools import read, write

print("Builtin:", read.name, "-", read.description)
print("Builtin:", write.name, "-", write.description)

Builtin: read - Read contents of a file, supporting line numbers and pagination.
Args:
    path: Absolute or relative path to the file.
    start_line: The starting line number (1-indexed) to read from.
    end_line: The ending line number (inclusive). Set to -1 to read to the end.
Builtin: write - Write or overwrite entirely the content of a file.
Args:
    path: Absolute or relative path to the file.
    content: The complete raw string content to write.
    overwrite: If false, fails if the file already exists.


---

## 2. Agent Class

Subclass `Agent`, configure in `setup()`, pass LLM at instantiation.

In [5]:
class MyAssistant(Agent):
    def setup(self):
        self.name = "assistant"
        self.description = "Personal assistant that remembers you"
        self.author = "Vaibhav"
        self.add_tools(multiply, calculator)

agent = MyAssistant(llm=llm)
print(agent.run("Hello! What can you do?"))

I can read files, write files, list directory contents, multiply numbers, and evaluate mathematical expressions. Just let me know what you need!



---

## 3. Memory & Conversations

Every agent has `self.memory`. When you call `agent.run()`, the conversation is **automatically saved** to `memory["messages"]`.

### 3A. Conversational memory — share info, agent remembers

In [6]:
# Share personal info with the agent
print(agent.run("Hi, my name is Vaibhav and my phone number is 9876543210"))

Okay, Vaibhav. I have recorded your name and phone number.


In [7]:
print(agent.run("I work at AgentComet as a developer"))

Okay, Vaibhav. I have recorded that you work at AgentComet as a developer.


In [8]:
print(agent.run("Remember: project deadline is March 15th"))

Okay, Vaibhav. I have recorded that the project deadline is March 15th.


In [8]:
# Now ask — the agent recalls from conversation history
print(agent.run("What is my name?"))

Vaibhav.


In [9]:
print(agent.run("What is my phone number?"))

Okay, Vaibhav. Your phone number is 9876543210.


In [9]:
# Check stored messages
msgs = agent.memory.get("messages", [])
print(f"\nConversation has {len(msgs)} messages:")
for msg in msgs:
    print(f"  [{msg['role']}] {msg['text'][:80]}")


Conversation has 8 messages:
  [user] Hello! What can you do?
  [agent] I can read files, write files, list directory contents, multiply numbers, and ev
  [user] Hi, my name is Vaibhav and my phone number is 9876543210
  [agent] Okay, Vaibhav. I have recorded your name and phone number.
  [user] I work at AgentComet as a developer
  [agent] Okay, Vaibhav. I have recorded that you work at AgentComet as a developer.
  [user] Remember: project deadline is March 15th
  [agent] Okay, Vaibhav. I have recorded that the project deadline is March 15th.


### 3B. Key-value memory — store structured data

In [10]:
# You can also store arbitrary key-value data
agent.memory.save("system_prompt", "You are a helpful personal assistant.")
agent.memory.save("preferences", {"theme": "dark", "lang": "en"})
agent.memory.save("notes", [
    "User prefers concise answers",
    "Deadline is March 15th"
])

print("All keys:", agent.memory.keys())
print("System prompt:", agent.memory.get("system_prompt"))
print("Notes:", agent.memory.get("notes"))
print()
print(agent.memory)  # Memory(N keys: [...])

All keys: ['messages', 'system_prompt', 'preferences', 'notes']
System prompt: You are a helpful personal assistant.
Notes: ['User prefers concise answers', 'Deadline is March 15th']

Memory(4 keys: ['messages', 'system_prompt', 'preferences', 'notes'])


---

## 4. State Persistence

Save memory snapshots — including full conversation history. Rollback anytime.

In [11]:
# Save current state (with all messages + data)
hash1 = agent.save_state()  # auto-hash
agent.save_state("after-intro")  # friendly name

[1226456a] State saved (4 keys)
[after-intro] State saved (4 keys)


'after-intro'

In [12]:
# Continue chatting — update info
print(agent.run("Actually, the deadline moved to March 20th"))
agent.memory.save("notes", ["Deadline updated to March 20th"])

agent.save_state("updated-deadline")

Okay, Vaibhav. I have recorded that the project deadline is March 20th.
[updated-deadline] State saved (4 keys)


'updated-deadline'

In [13]:
# View all checkpoints
agent.show_states()


States for 'assistant':
  [latest] 95a3e8a6 (updated-deadline)  2026-05-18 01:29:27  (4 keys)
           fd58f2cd (after-intro)  2026-05-18 01:29:19  (4 keys)
           1226456a  2026-05-18 01:29:19  (4 keys)
           761bf182  2026-05-17 17:20:11  (4 keys)
           c29814f8  2026-05-17 17:19:53  (4 keys)
           f154ac3d  2026-05-17 17:19:53  (4 keys)



[{'hash': '95a3e8a6', 'created_at': '2026-05-18 01:29:27', 'key_count': 4},
 {'hash': 'fd58f2cd', 'created_at': '2026-05-18 01:29:19', 'key_count': 4},
 {'hash': '1226456a', 'created_at': '2026-05-18 01:29:19', 'key_count': 4},
 {'hash': '761bf182', 'created_at': '2026-05-17 17:20:11', 'key_count': 4},
 {'hash': 'c29814f8', 'created_at': '2026-05-17 17:19:53', 'key_count': 4},
 {'hash': 'f154ac3d', 'created_at': '2026-05-17 17:19:53', 'key_count': 4}]

In [14]:
# Rollback to before deadline change
agent.load_state("after-intro")
print("After rollback:")
print("  Notes:", agent.memory.get("notes"))
print("  Messages:", len(agent.memory.get("messages", [])), "messages")

# Ask about deadline — should reflect the ORIGINAL date
print(agent.run("When is the project deadline?"))

Loaded state 'after-intro' (4 keys)
After rollback:
  Notes: ['User prefers concise answers', 'Deadline is March 15th']
  Messages: 8 messages
The project deadline is March 15th.


---

## 5. UAF Export & Load — Agent Remembers After Reload

Export to `.uaf` — conversation + memory auto-packed.  
Load later — agent picks up right where you left off.

In [15]:
# Reset to latest state (has all the info)
agent.load_state("updated-deadline")

# Export — everything auto-packed
agent.export("my_assistant.uaf")
print("Exported!")

Loaded state 'updated-deadline' (4 keys)
Exported AgentComet agent 'assistant' to my_assistant.uaf
Exported!


In [16]:
# Simulate a fresh session — load from file
loaded = load_agent("my_assistant.uaf")

print("Loaded agent type:", type(loaded))
print(f"Restored {len(loaded.memory.get('messages', []))} messages")
print("Restored notes:", loaded.memory.get("notes"))
print()

# Ask the loaded agent about info from BEFORE the export
print("--- Asking loaded agent about stored info ---")
print(loaded.run("What is my name?"))
print(loaded.run("What is my phone number?"))
print(loaded.run("When is the project deadline?"))

Loaded agent type: <class 'agentcomet.agents.factory.create_agent.<locals>.DynamicAgent'>
Restored 12 messages
Restored notes: ['Deadline updated to March 20th']

--- Asking loaded agent about stored info ---
Vaibhav.
I don’t have your phone number recorded. I only have your name, Vaibhav, and that you work at AgentComet as a developer.
The project deadline is March 20th.


In [17]:
# Cleanup
os.remove("my_assistant.uaf")

---

## 6. Local Server Interaction

Sync your agents with a locally hosted AgentComet server effortlessly. The SDK automates generating the portable UAF and determining the next semantic version.


In [ ]:
from agentcomet import Settings, Agent

# Configure connection programmatically from loaded environment variables
Settings.init(
    AGENTCOMET_URL=os.getenv("AGENTCOMET_LOCAL_URL", "http://localhost:3451"),
    AGENTCOMET_KEY=os.getenv("AGENTCOMET_LOCAL_KEY")
)

try:
    # Push the agent. UAF is created dynamically, and version is auto-incremented!
    push_res = agent.push(repo="assistant", version="auto", create=True)
    print("Push response:", push_res)
    
    # Pull the agent from the server. It downloads the UAF and loads it instantly!
    downloaded_agent = Agent.pull(repo="assistant", version="latest")
    print("Downloaded agent:", downloaded_agent.name)
except Exception as e:
    print("Local Server Interaction Error:", e)


Exported AgentComet agent 'assistant' to C:\Users\Vaibh\AppData\Local\Temp\tmpbpuo9f2i.uaf
Local Server Interaction Error: Hub push failed with HTTP 400: {"error":"name, description, version, and artifact are required."}


---

## 7. LLM Providers

Pass LLM instances directly to agents.

In [ ]:
from agentcomet.models import Ollama, OpenAIChat, Gemini, Anthropic, OpenRouter, Perplexity

# Ollama (local)
ollama = Ollama(model="gemma3:4b")
result = ollama.generate("What is the capital of France?")
print("Direct call:", result[:100])

# Other providers (require API keys)
# openai = OpenAIChat(model="gpt-4o")
# gemini = Gemini(model="gemini-1.5-flash")
# claude = Anthropic(model="claude-3-5-sonnet")

---

## 8. AgentComet Hub

Sync your agents with the cloud-hosted AgentComet Hub (`ac.defaultloop.com`). You can use the local server commands by pointing the URL to the hub.


In [15]:
from agentcomet import Settings, Agent

# Configure connection to the public Hub from environment variables
Settings.init(
    AGENTCOMET_URL=os.getenv("AGENTCOMET_URL", "https://ac.defaultloop.com"),
    AGENTCOMET_KEY=os.getenv("AGENTCOMET_KEY")
)

try:
    # Push the agent to the Hub.
    push_res = agent.push(repo="vhx/assistant", version="auto", create=True)
    print("Push response:", push_res)
    
    # Pull the agent from the Hub.
    downloaded_agent = Agent.pull(repo="vhx/assistant", version="latest")
    print("Downloaded agent:", downloaded_agent.name)
except Exception as e:
    print("Hub Interaction Error:", e)


Exported AgentComet agent 'assistant' to C:\Users\Vaibh\AppData\Local\Temp\tmp_hjgy26d.uaf
Hub Interaction Error: Hub push failed with HTTP 404: ...


---

## Cleanup

In [ ]:
if os.path.exists(".agentcomet"):
    shutil.rmtree(".agentcomet")
    print("Cleaned up .agentcomet/")

---

## Summary

| Feature | API | Description |
|---------|-----|-------------|
| **Tools** | `@tool` | Auto ToolSpec from functions |
| **Agent** | `class MyAgent(Agent)` | Full control via `setup()` |
| **Declarative** | `create_agent(...)` | One-liner agents |
| **Auto Memory** | `agent.run("...")` | Chat auto-saved to messages |
| **Key-Value** | `memory.save(k, v)` | Store anything |
| **Save State** | `save_state("name")` | Named or hashed checkpoint |
| **Load State** | `load_state("name")` | Rollback to any checkpoint |
| **Export** | `agent.export("file.uaf")` | Memory auto-packed |
| **Load** | `load_agent("file.uaf")` | Agent remembers everything |
| **Local Sync** | `push_local()`, `pull_local()` | Sync with local server |
| **LLM** | `Ollama(model=...)` | 6+ providers, pass directly |